In [1]:
!wget https://www.lamsade.dauphine.fr/~cazenave/project2026.zip
!unzip project2026.zip
!ls -l

--2026-03-29 23:30:33--  https://www.lamsade.dauphine.fr/~cazenave/project2026.zip
Resolving www.lamsade.dauphine.fr (www.lamsade.dauphine.fr)... 193.48.71.250
Connecting to www.lamsade.dauphine.fr (www.lamsade.dauphine.fr)|193.48.71.250|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 138578548 (132M) [application/zip]
Saving to: ‘project2026.zip’

project2026.zip     100%[===================>] 132.16M  19.5MB/s    in 7.4s    

2026-03-29 23:30:41 (17.8 MB/s) - ‘project2026.zip’ saved [138578548/138578548]

Archive:  project2026.zip
  inflating: games.data              
  inflating: golois.cpython-312-x86_64-linux-gnu.so  
total 665448
-rw-r--r-- 1 root root 542497580 Oct  7  2022 games.data
-rwxr-xr-x 1 root root    284672 Oct  1 15:09 golois.cpython-312-x86_64-linux-gnu.so
---------- 1 root root     52359 Mar 29 23:30 __notebook__.ipynb
-rw-r--r-- 1 root root 138578548 Oct  1 20:02 project2026.zip


In [2]:
!pip install wandb weave

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 955.0/955.0 kB 15.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.9/45.9 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.9/89.9 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 56.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.9/52.9 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.3/207.3 kB 12.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 88.6 MB/s eta 0:00:00


In [3]:
import wandb
from kaggle_secrets import UserSecretsClient

WANDB_KEY = UserSecretsClient().get_secret("monitor GO")
wandb.login(key=WANDB_KEY)

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: rouibiamine7 (rouibiamine7-universit-paris-dauphine-psl) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

# Base CODE

In [4]:
# ═══════════════════════════════════════════════════════════════════
#  CELLULE BASE — Infrastructure partagée | Projet Go 19×19
#  À exécuter UNE SEULE FOIS avant n'importe quel modèle
# ═══════════════════════════════════════════════════════════════════

import os
import gc
import csv
import random
import shutil
import datetime
import numpy as np
import tensorflow as tf
import tensorflow.keras as keras
from tensorflow.keras import layers, regularizers
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

# ── Juste après les imports ───────────────────────────────────────

# ← AJOUTÉ : mixed precision — FP16 sur Tensor Cores (T4 / P100)
keras.mixed_precision.set_global_policy('float32')

# ← AJOUTÉ : détection multi-GPU
strategy = tf.distribute.MirroredStrategy()
NUM_GPUS  = strategy.num_replicas_in_sync
print(f"GPUs détectés : {NUM_GPUS}")


import golois

# ───────────────────────────────────────────────────────────────────
# CONSTANTES GLOBALES
# ───────────────────────────────────────────────────────────────────
DRIVE_DIR  = '/kaggle/working/go_project3'
RUNS_DIR   = 'runs'
PLANES     = 31
MOVES      = 361
N          = 50000

os.makedirs(DRIVE_DIR, exist_ok=True)
os.makedirs(RUNS_DIR,  exist_ok=True)

print(f"TensorFlow : {tf.__version__}")
print(f"GPUs       : {tf.config.list_physical_devices('GPU')}")

# ───────────────────────────────────────────────────────────────────
# SEEDS
# ───────────────────────────────────────────────────────────────────
def set_seeds(seed):
    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)
    os.environ['PYTHONHASHSEED']       = str(seed)
    os.environ['TF_DETERMINISTIC_OPS'] = '1'

set_seeds(42)
# ───────────────────────────────────────────────────────────────────
# DONNÉES — init une seule fois, réutilisées par tous les modèles
# ───────────────────────────────────────────────────────────────────
print("\nInitialisation des données...", flush=True)

# Train (écrasé à chaque getBatch)
input_data = np.random.randint(2, size=(N, 19, 19, PLANES)).astype('float32')
policy     = keras.utils.to_categorical(
                np.random.randint(MOVES, size=(N,)), num_classes=MOVES
             ).astype('float32')
value      = np.random.randint(2, size=(N,)).astype('float32')
end        = np.random.randint(2, size=(N, 19, 19, 2)).astype('float32')
groups     = np.zeros((N, 19, 19, 1), dtype='float32')

# Val set fixe — rempli une seule fois, jamais retouché
val_input  = np.zeros((N, 19, 19, PLANES), dtype='float32')
val_policy = np.zeros((N, MOVES),          dtype='float32')
val_value  = np.zeros((N,),                dtype='float32')
val_end    = np.zeros((N, 19, 19, 2),      dtype='float32')

golois.getValidation(val_input, val_policy, val_value, val_end)
print("✅ Val set fixe chargé — ne sera plus modifié\n")

# ───────────────────────────────────────────────────────────────────
# HELPERS
# ───────────────────────────────────────────────────────────────────
def make_run_dir(config):
    """Crée le dossier de run horodaté et écrit config.txt."""
    timestamp = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
    run_name  = (f"model{config['model_id']}_{config['model_name']}"
                 f"_lr{config['lr']}_seed{config['seed']}_{timestamp}")
    run_dir   = os.path.join(RUNS_DIR, run_name)
    os.makedirs(run_dir, exist_ok=True)

    with open(os.path.join(run_dir, 'config.txt'), 'w') as f:
        for k, v in config.items():
            f.write(f"{k} = {v}\n")
        f.write(f"\ntf_version = {tf.__version__}\n")
        f.write(f"gpu        = {tf.config.list_physical_devices('GPU')}\n")
        f.write(f"timestamp  = {timestamp}\n")

    print(f"\n{'='*60}")
    print(f"  RUN : {run_name}")
    print(f"  DIR : {run_dir}")
    print(f"{'='*60}\n")
    return run_dir, run_name, timestamp


def make_csv(run_dir):
    """Crée les fichiers CSV train et val avec leurs headers."""
    csv_fields = ['epoch', 'loss', 'policy_loss', 'value_loss',
                  'policy_categorical_accuracy', 'value_mae']
    val_fields = ['epoch', 'total_loss', 'policy_loss', 'value_loss',
                  'policy_categorical_accuracy', 'value_mae']

    csv_path = os.path.join(run_dir, 'history.csv')
    val_path = os.path.join(run_dir, 'val_results.csv')

    with open(csv_path, 'w', newline='') as f:
        csv.DictWriter(f, fieldnames=csv_fields).writeheader()
    with open(val_path, 'w', newline='') as f:
        csv.DictWriter(f, fieldnames=val_fields).writeheader()

    return csv_path, csv_fields, val_path, val_fields


def plot_curves(history_log, run_dir, run_name, epoch):
    """Sauvegarde les courbes train intermédiaires."""
    epochs_range = history_log['epoch']
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    fig.suptitle(run_name, fontsize=9)

    axes[0].plot(epochs_range, history_log['loss'], 'b-o', markersize=3)
    axes[0].set_title('Total Loss'); axes[0].set_xlabel('Epoch'); axes[0].grid(True, alpha=0.3)

    axes[1].plot(epochs_range, history_log['policy_categorical_accuracy'], 'g-o', markersize=3)
    axes[1].set_title('Policy Accuracy'); axes[1].set_xlabel('Epoch'); axes[1].grid(True, alpha=0.3)

    axes[2].plot(epochs_range, history_log['value_mae'], 'r-o', markersize=3)
    axes[2].set_title('Value MAE'); axes[2].set_xlabel('Epoch'); axes[2].grid(True, alpha=0.3)

    plt.tight_layout()
    plot_path = os.path.join(run_dir, f'curves_epoch{epoch}.png')
    plt.savefig(plot_path, dpi=120)
    plt.close()
    print(f"  [PLOT] → {plot_path}")


def save_final_plots(history_log, run_dir):
    """Sauvegarde les graphiques rapport final."""
    plt.style.use('seaborn-v0_8-whitegrid')
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
    epochs = history_log['epoch']

    ax1.plot(epochs, history_log['policy_categorical_accuracy'],
             color='#2ca02c', label='Policy Accuracy')
    ax1.set_title('Évolution de la précision (Policy)')
    ax1.set_xlabel('Époques'); ax1.set_ylabel('Précision'); ax1.legend()

    ax2.plot(epochs, history_log['loss'], color='#d62728', label='Total Loss')
    ax2.set_title('Évolution de la perte (Loss)')
    ax2.set_xlabel('Époques'); ax2.set_ylabel('Perte'); ax2.legend()

    plt.tight_layout()
    plt.savefig(os.path.join(run_dir, 'report_visuals.png'), dpi=300)
    plt.close()
    print("✅ report_visuals.png sauvegardé")


def drive_sync_light(run_dir, run_name):
    """Sync léger en cours de run : CSV + PNG + TXT uniquement."""
    drive_run_dir = os.path.join(DRIVE_DIR, run_name)
    os.makedirs(drive_run_dir, exist_ok=True)
    for fname in os.listdir(run_dir):
        if fname.endswith(('.csv', '.png', '.txt')):
            shutil.copy(os.path.join(run_dir, fname),
                        os.path.join(drive_run_dir, fname))
    print(f"  [DRIVE] Sync léger → {drive_run_dir}")


def drive_sync_full(run_dir, run_name):
    """Sync complet en fin de run (inclut les .keras)."""
    drive_run_dir = os.path.join(DRIVE_DIR, run_name)
    shutil.copytree(run_dir, drive_run_dir, dirs_exist_ok=True)

    global_csv = os.path.join(RUNS_DIR, 'all_runs_summary.csv')
    if os.path.exists(global_csv):
        shutil.copy(global_csv, os.path.join(DRIVE_DIR, 'all_runs_summary.csv'))
    print(f"  [DRIVE] Sync final → {drive_run_dir}")


def update_global_summary(config, run_name, timestamp,
                           total_params, best_loss, best_policy_acc, epochs_run):
    """Ajoute une ligne au CSV global all_runs_summary.csv."""
    global_csv    = os.path.join(RUNS_DIR, 'all_runs_summary.csv')
    global_fields = ['run_name', 'model_id', 'model_name', 'params',
                     'best_loss', 'best_policy_acc',
                     'lr', 'seed', 'epochs_run', 'timestamp']
    file_exists = os.path.exists(global_csv)

    with open(global_csv, 'a', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=global_fields)
        if not file_exists:
            writer.writeheader()
        writer.writerow({
            'run_name'        : run_name,
            'model_id'        : config['model_id'],
            'model_name'      : config['model_name'],
            'params'          : total_params,
            'best_loss'       : round(best_loss, 6),
            'best_policy_acc' : round(best_policy_acc, 6),
            'lr'              : config['lr'],
            'seed'            : config['seed'],
            'epochs_run'      : epochs_run,
            'timestamp'       : timestamp,
        })


def print_final_summary(config, run_dir, run_name,
                         best_loss, best_policy_acc, total_params, epochs_run):
    print(f"\n{'='*60}")
    print(f"  RUN TERMINÉ : {run_name}")
    print(f"  Drive       : {DRIVE_DIR}/{run_name}")
    print(f"  Best loss        : {best_loss:.4f}")
    print(f"  Best policy acc  : {best_policy_acc:.4f}")
    print(f"  Epochs réelles   : {epochs_run}/{config['epochs']}")
    print(f"  Params           : {total_params:,}")
    print(f"  Outputs          : {run_dir}/")
    print(f"    ├── config.txt")
    print(f"    ├── history.csv          (train, toutes les epochs)")
    print(f"    ├── val_results.csv      (val fixe, tous les 10 epochs)")
    print(f"    ├── best_model.keras")
    print(f"    ├── logs/                (TensorBoard)")
    print(f"    ├── curves_epoch[n].png  (plots intermédiaires)")
    print(f"    └── report_visuals.png   (graphiques rapport)")
    print(f"  Global : {RUNS_DIR}/all_runs_summary.csv")
    print(f"{'='*60}\n")

# ───────────────────────────────────────────────────────────────────
# FONCTION D'ENTRAÎNEMENT UNIVERSELLE
# ───────────────────────────────────────────────────────────────────
def train_model(build_fn, config):
    """
    Entraîne n'importe quel modèle compilé avec le socle standard.
    build_fn(config) est appelé dans strategy.scope().
    """
    set_seeds(config['seed'])
    with strategy.scope():
        model = build_fn(config)

    # ── Setup run ────────────────────────────────────────────────────
    run_dir, run_name, timestamp = make_run_dir(config)
    wandb.init(
        project = "go-19x19",
        name    = run_name,
        config  = config,
        reinit  = True
    )
    csv_path, csv_fields, val_path, val_fields = make_csv(run_dir)
    history_log = {k: [] for k in csv_fields}

    checkpoint_path = os.path.join(run_dir, 'best_model.keras')
    log_dir         = os.path.join(run_dir, 'logs')

    # ── Callbacks ────────────────────────────────────────────────────
    callbacks = [
       keras.callbacks.ReduceLROnPlateau(
            monitor='loss', factor=0.5,
            patience=config['patience'], mode='min',
            min_lr=1e-6, min_delta=1e-4,  # ← AJOUTÉ : évite réductions LR sur du bruit
            verbose=1
        ),
    ]

    tb_writer = tf.summary.create_file_writer(log_dir)

    # ── Boucle ───────────────────────────────────────────────────────
    best_loss          = float('inf')
    best_policy_acc    = 0.0
    early_stop_counter = 0

    for i in range(1, config['epochs'] + 1):
        print(f"\n{'─'*50}")
        print(f"  Epoch {i}/{config['epochs']}")
        print(f"{'─'*50}")

        golois.getBatch(input_data, policy, value, end, groups, i * N)

        hist = model.fit(
            input_data, [policy, value],
            initial_epoch=i - 1,
            epochs=i,
            batch_size=config['batch'],
            callbacks=callbacks,
            verbose=1
        )

        # ── Métriques ────────────────────────────────────────────────
        h = hist.history
        row = {
            'epoch'                       : i,
            'loss'                        : h['loss'][0],
            'policy_loss'                 : h['policy_loss'][0],
            'value_loss'                  : h['value_loss'][0],
            'policy_categorical_accuracy' : h['policy_categorical_accuracy'][0],
            'value_mae'                   : h['value_mae'][0],
        }

        with open(csv_path, 'a', newline='') as f:
            csv.DictWriter(f, fieldnames=csv_fields).writerow(row)
        for k, v in row.items():
            history_log[k].append(v)

        if row['policy_categorical_accuracy'] > best_policy_acc:
            best_policy_acc = row['policy_categorical_accuracy']

        print(f"  loss={row['loss']:.4f} | "
              f"policy_acc={row['policy_categorical_accuracy']:.4f} | "
              f"value_mae={row['value_mae']:.4f}")

        # ── TensorBoard ──────────────────────────────────────────────
        with tb_writer.as_default():
            tf.summary.scalar('loss/total',  row['loss'],                        step=i)
            tf.summary.scalar('loss/policy', row['policy_loss'],                 step=i)
            tf.summary.scalar('loss/value',  row['value_loss'],                  step=i)
            tf.summary.scalar('acc/policy',  row['policy_categorical_accuracy'], step=i)
            tf.summary.scalar('mae/value',   row['value_mae'],                   step=i)
        tb_writer.flush()

        # ── Val loss (chaque epoch — pour checkpoint et early stopping) ─
        val = model.evaluate(
            val_input, [val_policy, val_value],
            verbose=0, batch_size=config['batch']
        )

        current_val_loss       = val[0]
        current_val_policy_acc = val[3]

        wandb.log({
            'epoch'            : i,
            'train_loss'       : row['loss'],
            'train_policy_acc' : row['policy_categorical_accuracy'],
            'train_value_mae'  : row['value_mae'],
            'val_loss'         : current_val_loss,
            'val_policy_acc'   : current_val_policy_acc,
            'lr'               : float(keras.backend.get_value(
                                     model.optimizer.learning_rate))
        })

        # ── Early stopping ───────────────────────────────────────────────
        if current_val_loss < best_loss:
            best_loss          = current_val_loss
            early_stop_counter = 0
            model.save(checkpoint_path)  # ← save manuel basé sur val_loss
            print(f"  [CHECKPOINT] val_loss → {best_loss:.4f} — modèle sauvegardé")
        else:
            early_stop_counter += 1
            print(f"  [EarlyStopping] patience {early_stop_counter}/{config['patience']}")
            if early_stop_counter >= config['patience']:
                print(f"  [EarlyStopping] Arrêt à l'époque {i}.")
                break

        if i % 5 == 0:
            gc.collect()

        # ── Val + plot + sync tous les 10 epochs ─────────────────────
        if i % 10 == 0:
            print(f"\n  [VAL epoch {i}] total={current_val_loss:.4f} | "
                  f"policy_acc={val[3]:.4f} | value_mae={val[4]:.4f}")

            with open(val_path, 'a', newline='') as f:
                csv.DictWriter(f, fieldnames=val_fields).writerow({
                    'epoch'                       : i,
                    'total_loss'                  : current_val_loss,
                    'policy_loss'                 : val[1],
                    'value_loss'                  : val[2],
                    'policy_categorical_accuracy' : val[3],
                    'value_mae'                   : val[4],
                })

            plot_curves(history_log, run_dir, run_name, i)
            drive_sync_light(run_dir, run_name)

    tb_writer.close()

    # ── Fin de run ───────────────────────────────────────────────────
    epochs_run = len(history_log['epoch'])
    save_final_plots(history_log, run_dir)
    update_global_summary(config, run_name, timestamp,
                          model.count_params(), best_loss, best_policy_acc, epochs_run)
    drive_sync_full(run_dir, run_name)
    print_final_summary(config, run_dir, run_name,
                        best_loss, best_policy_acc, model.count_params(), epochs_run)
    wandb.finish()
    return history_log, best_loss, best_policy_acc


print("✅ Cellule BASE chargée — prêt pour les modèles")

2026-03-29 23:31:01.935121: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1774827062.630920      24 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1774827062.811025      24 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1774827063.830456      24 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774827063.830501      24 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774827063.830504      24 computation_placer.cc:177] computation placer alr

INFO:tensorflow:Using MirroredStrategy with devices ('/job:localhost/replica:0/task:0/device:GPU:0', '/job:localhost/replica:0/task:0/device:GPU:1')
GPUs détectés : 2
TensorFlow : 2.19.0
GPUs       : [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU'), PhysicalDevice(name='/physical_device:GPU:1', device_type='GPU')]

Initialisation des données...


I0000 00:00:1774827102.393710      24 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13757 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1774827102.399837      24 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13757 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


✅ Val set fixe chargé — ne sera plus modifié

✅ Cellule BASE chargée — prêt pour les modèles


r.shape = (50000, 19, 19, 31)
nbExamples = 50000
nbPositionsSGF = 29425326
nbPositionsSGF = 29425326
generating validation.data


# M3 : CNN + BatchNorm

In [5]:
# ═══════════════════════════════════════════════════════════════════
#  CELLULE M3 — CNN + BatchNorm | Projet Go 19×19
#  Identique M2 + BN après chaque Conv
#  dense_head : 64→66 pour maximiser sous 100k
#  ≈ 99 752 params
#
#  Budget :
#    M2 base         = 97 388
#    BN ×3 (64/layer)=    192  → 97 580
#    dense_head 64→66=  2 172  → 99 752  ✓ < 100k
#
#  ⚠️  Nécessite la cellule BASE exécutée au préalable
# ═══════════════════════════════════════════════════════════════════

config = {
    'model_id'   : 3,
    'model_name' : 'cnn_batchnorm',
    'planes'     : PLANES,
    'moves'      : MOVES,
    'N'          : N,
    'epochs'     : 300,
    'batch'      : 128 * NUM_GPUS,
    'lr'         : 0.001,
    'l2'         : 0.0001,
    'seed'       : 42,
    'max_params' : 100_000,
    'patience'   : 20,
    'filters'    : 32,
    'dense_head' : 66,          # ← 64 en M2, monté à 66 pour maximiser
}

def build_model(config):
    reg = regularizers.l2(config['l2'])
    f   = config['filters']
    dh  = config['dense_head']

    inp = keras.Input(shape=(19, 19, config['planes']), name='board')

    # ── Backbone : Conv + BN après chaque couche ─────────────────────
    x = layers.Conv2D(f, (3, 3), padding='same', use_bias=False,
                      kernel_regularizer=reg)(inp)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)

    x = layers.Conv2D(f, (3, 3), padding='same', use_bias=False,
                      kernel_regularizer=reg)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)

    x = layers.Conv2D(f, (3, 3), padding='same', use_bias=False,
                      kernel_regularizer=reg)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)

    # ── Policy head ──────────────────────────────────────────────────
    p = layers.Conv2D(1, (1, 1), activation='relu', kernel_regularizer=reg)(x)
    p = layers.Flatten()(p)
    p = layers.Dense(dh, activation='relu', kernel_regularizer=reg)(p)
    policy_head = layers.Dense(config['moves'], activation='softmax',
                                name='policy', kernel_regularizer=reg,
                                dtype='float32')(p)

    # ── Value head ───────────────────────────────────────────────────
    v = layers.Conv2D(1, (1, 1), activation='relu', kernel_regularizer=reg)(x)
    v = layers.Flatten()(v)
    v = layers.Dense(dh, activation='relu', kernel_regularizer=reg)(v)
    value_head = layers.Dense(1, activation='sigmoid',
                               name='value', kernel_regularizer=reg,
                               dtype='float32')(v)

    model = keras.Model(inputs=inp, outputs=[policy_head, value_head])
    model.summary()

    total_params = model.count_params()
    print(f"\n>>> Total params : {total_params:,}")
    assert total_params < config['max_params'], \
        f"CONTRAINTE VIOLÉE : {total_params:,} > {config['max_params']:,} !"
    print(f">>> Contrainte < {config['max_params']:,} params : ✓\n")

    model.compile(
        optimizer=keras.optimizers.Adam(
            learning_rate=config['lr'],
            clipnorm=1.0              # ← AJOUTÉ : évite NaN explosion en FP16
        ),
        loss={'policy': 'categorical_crossentropy', 'value': 'mse'},
        loss_weights={'policy': 1.0, 'value': 1.0},
        metrics={'policy': 'categorical_accuracy', 'value': 'mae'}
    )
    return model

# ── Lancement ────────────────────────────────────────────────────────
history_m3, best_loss_m3, best_acc_m3 = train_model(build_model, config)

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ board (InputLayer)  │ (None, 19, 19,    │          0 │ -                 │
│                     │ 31)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d (Conv2D)     │ (None, 19, 19,    │      8,928 │ board[0][0]       │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalization │ (None, 19, 19,    │        128 │ conv2d[0][0]      │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation          │ (None, 19, 19,    │          0 │ batch_normalizat… │
│ (Activation)        │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_1 (Conv2D)   │ (None, 19, 19,    │      9,216 │ activation[0][0]  │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 19, 19,    │        128 │ conv2d_1[0][0]    │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_1        │ (None, 19, 19,    │          0 │ batch_normalizat… │
│ (Activation)        │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_2 (Conv2D)   │ (None, 19, 19,    │      9,216 │ activation_1[0][… │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 19, 19,    │        128 │ conv2d_2[0][0]    │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_2        │ (None, 19, 19,    │          0 │ batch_normalizat… │
│ (Activation)        │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_3 (Conv2D)   │ (None, 19, 19, 1) │         33 │ activation_2[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_4 (Conv2D)   │ (None, 19, 19, 1) │         33 │ activation_2[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flatten (Flatten)   │ (None, 361)       │          0 │ conv2d_3[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flatten_1 (Flatten) │ (None, 361)       │          0 │ conv2d_4[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 66)        │     23,892 │ flatten[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 66)        │     23,892 │ flatten_1[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ policy (Dense)      │ (None, 361)       │     24,187 │ dense[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ value (Dense)       │ (None, 1)         │         67 │ dense_1[0][0]     │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 99,848 (390.03 KB)

 Trainable params: 99,656 (389.28 KB)

 Non-trainable params: 192 (768.00 B)

...

2026-03-30 01:29:45.083214: E tensorflow/core/framework/node_def_util.cc:680] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_16}}
2026-03-30 01:29:51.421849: E tensorflow/core/framework/node_def_util.cc:680] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),

  [EarlyStopping] patience 20/20
  [EarlyStopping] Arrêt à l'époque 174.


wandb: updating run metadata


✅ report_visuals.png sauvegardé
  [DRIVE] Sync final → /kaggle/working/go_project3/model3_cnn_batchnorm_lr0.001_seed42_20260329_233218

  RUN TERMINÉ : model3_cnn_batchnorm_lr0.001_seed42_20260329_233218
  Drive       : /kaggle/working/go_project3/model3_cnn_batchnorm_lr0.001_seed42_20260329_233218
  Best loss        : 3.6282
  Best policy acc  : 0.3402
  Epochs réelles   : 174/300
  Params           : 99,848
  Outputs          : runs/model3_cnn_batchnorm_lr0.001_seed42_20260329_233218/
    ├── config.txt
    ├── history.csv          (train, toutes les epochs)
    ├── val_results.csv      (val fixe, tous les 10 epochs)
    ├── best_model.keras
    ├── logs/                (TensorBoard)
    ├── curves_epoch[n].png  (plots intermédiaires)
    └── report_visuals.png   (graphiques rapport)
  Global : runs/all_runs_summary.csv



wandb: 
wandb: Run history:
wandb:            epoch ▁▁▂▂▂▂▃▃▃▃▃▃▃▃▃▄▄▄▄▄▅▅▅▅▆▆▆▆▆▆▇▇▇▇▇▇████
wandb:               lr ▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
wandb:       train_loss █▆▅▄▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
wandb: train_policy_acc ▁▂▆▆▇▇▇▇▇▇██████████████████████████████
wandb:  train_value_mae ▃▆▆▆▅▇▇█▇▃▃▅▆█▅▄▃▅▇▃▄▅▁▄▄▃█▇▆▄▅▁▅▅▄▅▅▅▃▂
wandb:         val_loss █▃▂▂▂▂▁▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
wandb:   val_policy_acc ▁▃▅▇▇▇▇▇▇▇██████████████████████████████
wandb: 
wandb: Run summary:
wandb:            epoch 174
wandb:               lr 0.001
wandb:       train_loss 3.62139
wandb: train_policy_acc 0.33402
wandb:  train_value_mae 0.28582
wandb:         val_loss 3.6323
wandb:   val_policy_acc 0.33512
wandb: 
wandb: 🚀 View run model3_cnn_batchnorm_lr0.001_seed42_20260329_233218 at: https://wandb.ai/rouibiamine7-universit-paris-dauphine-psl/go-19x19/runs/nc3qiofa
wandb: ⭐️ View project at: https://wandb.ai/rouibiamine7-universit-paris-dauphine-psl/go-19x19
wandb: Synced

# M4 : ResNet Tiny

In [6]:
# ═══════════════════════════════════════════════════════════════════
#  CELLULE M4 — ResNet Tiny | Projet Go 19×19
#  Conv(24) → [ResBlock(24)]×3 → dual head(fc=56)
#
#  Budget :
#    Initial Conv(24,3×3)           =  6 720
#    ResBlock×3 (BN, no skip params)= 31 680
#    Policy Conv(1)+Dense(56)+out   = 40 874
#    Value  Conv(1)+Dense(56)+out   = 20 354
#    ─────────────────────────────────────────
#    Total                          = 99 628  ✓ < 100k
#    (fc=48 décrit → 90 940 | fc=57 → 100 786 ❌)
#
#  ⚠️  Nécessite la cellule BASE exécutée au préalable
# ═══════════════════════════════════════════════════════════════════

config = {
    'model_id'   : 4,
    'model_name' : 'resnet_tiny',
    'planes'     : PLANES,
    'moves'      : MOVES,
    'N'          : N,
    'epochs'     : 300,
    'batch'      : 128 * NUM_GPUS,
    'lr'         : 0.001,
    'l2'         : 0.0001,
    'seed'       : 42,
    'max_params' : 100_000,
    'patience'   : 20,
    'filters'    : 24,
    'n_blocks'   : 3,
    'dense_head' : 56,          # ← 48 en description, monté à 56 pour maximiser
}

def resblock(x, filters, reg):
    """Conv(no_bias)+BN+ReLU+Conv(no_bias)+BN + skip identity."""
    shortcut = x
    x = layers.Conv2D(filters, (3, 3), padding='same', use_bias=False,
                      kernel_regularizer=reg)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.Conv2D(filters, (3, 3), padding='same', use_bias=False,
                      kernel_regularizer=reg)(x)
    x = layers.BatchNormalization(gamma_initializer='zeros')(x)
    x = layers.Add()([x, shortcut])   # skip connection — 0 params
    x = layers.Activation('relu')(x)
    return x

def build_model(config):
    reg = regularizers.l2(config['l2'])
    f   = config['filters']
    dh  = config['dense_head']

    inp = keras.Input(shape=(19, 19, config['planes']), name='board')

    # ── Stem ─────────────────────────────────────────────────────────
    x = layers.Conv2D(f, (3, 3), padding='same', use_bias=False,
                      kernel_regularizer=reg)(inp)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)

    # ── ResBlocks ────────────────────────────────────────────────────
    for _ in range(config['n_blocks']):
        x = resblock(x, f, reg)

    # ── Policy head ──────────────────────────────────────────────────
    p = layers.Conv2D(1, (1, 1), activation='relu', kernel_regularizer=reg)(x)
    p = layers.Flatten()(p)
    p = layers.Dense(dh, activation='relu', kernel_regularizer=reg)(p)
    policy_head = layers.Dense(config['moves'], activation='softmax',
                                name='policy', kernel_regularizer=reg,
                                dtype='float32')(p)

    # ── Value head ───────────────────────────────────────────────────
    v = layers.Conv2D(1, (1, 1), activation='relu', kernel_regularizer=reg)(x)
    v = layers.Flatten()(v)
    v = layers.Dense(dh, activation='relu', kernel_regularizer=reg)(v)
    value_head = layers.Dense(1, activation='sigmoid',
                               name='value', kernel_regularizer=reg,
                               dtype='float32')(v)

    model = keras.Model(inputs=inp, outputs=[policy_head, value_head])
    model.summary()

    total_params = model.count_params()
    print(f"\n>>> Total params : {total_params:,}")
    assert total_params < config['max_params'], \
        f"CONTRAINTE VIOLÉE : {total_params:,} > {config['max_params']:,} !"
    print(f">>> Contrainte < {config['max_params']:,} params : ✓\n")

    model.compile(
        optimizer=keras.optimizers.Adam(
            learning_rate=config['lr'],
            clipnorm=1.0              # ← AJOUTÉ : évite NaN explosion en FP16
        ),
        loss={'policy': 'categorical_crossentropy', 'value': 'mse'},
        loss_weights={'policy': 1.0, 'value': 1.0},
        metrics={'policy': 'categorical_accuracy', 'value': 'mae'}
    )
    return model

# ── Lancement ────────────────────────────────────────────────────────
history_m4, best_loss_m4, best_acc_m4 = train_model(build_model, config)

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ board (InputLayer)  │ (None, 19, 19,    │          0 │ -                 │
│                     │ 31)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_5 (Conv2D)   │ (None, 19, 19,    │      6,696 │ board[0][0]       │
│                     │ 24)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 19, 19,    │         96 │ conv2d_5[0][0]    │
│ (BatchNormalizatio… │ 24)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_3        │ (None, 19, 19,    │          0 │ batch_normalizat… │
│ (Activation)        │ 24)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_6 (Conv2D)   │ (None, 19, 19,    │      5,184 │ activation_3[0][… │
│                     │ 24)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 19, 19,    │         96 │ conv2d_6[0][0]    │
│ (BatchNormalizatio… │ 24)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_4        │ (None, 19, 19,    │          0 │ batch_normalizat… │
│ (Activation)        │ 24)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_7 (Conv2D)   │ (None, 19, 19,    │      5,184 │ activation_4[0][… │
│                     │ 24)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 19, 19,    │         96 │ conv2d_7[0][0]    │
│ (BatchNormalizatio… │ 24)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add (Add)           │ (None, 19, 19,    │          0 │ batch_normalizat… │
│                     │ 24)               │            │ activation_3[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_5        │ (None, 19, 19,    │          0 │ add[0][0]         │
│ (Activation)        │ 24)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_8 (Conv2D)   │ (None, 19, 19,    │      5,184 │ activation_5[0][… │
│                     │ 24)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 19, 19,    │         96 │ conv2d_8[0][0]    │
│ (BatchNormalizatio… │ 24)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_6        │ (None, 19, 19,    │          0 │ batch_normalizat… │
│ (Activation)        │ 24)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_9 (Conv2D)   │ (None, 19, 19,    │      5,184 │ activation_6[0][… │
│                     │ 24)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 19, 19,    │         96 │ conv2d_9[0][0]    │
│ (BatchNormalizatio… │ 24)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_1 (Add)         │ (None, 19, 19,    │          0 │ batch_normalizat

 Total params: 99,700 (389.45 KB)

 Trainable params: 99,364 (388.14 KB)

 Non-trainable params: 336 (1.31 KB)

...

2026-03-30 03:35:36.178955: E tensorflow/core/framework/node_def_util.cc:680] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_16}}
2026-03-30 03:35:43.033187: E tensorflow/core/framework/node_def_util.cc:680] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),

  [EarlyStopping] patience 20/20
  [EarlyStopping] Arrêt à l'époque 176.


wandb: updating run metadata


✅ report_visuals.png sauvegardé
  [DRIVE] Sync final → /kaggle/working/go_project3/model4_resnet_tiny_lr0.001_seed42_20260330_012956

  RUN TERMINÉ : model4_resnet_tiny_lr0.001_seed42_20260330_012956
  Drive       : /kaggle/working/go_project3/model4_resnet_tiny_lr0.001_seed42_20260330_012956
  Best loss        : 3.5021
  Best policy acc  : 0.3557
  Epochs réelles   : 176/300
  Params           : 99,700
  Outputs          : runs/model4_resnet_tiny_lr0.001_seed42_20260330_012956/
    ├── config.txt
    ├── history.csv          (train, toutes les epochs)
    ├── val_results.csv      (val fixe, tous les 10 epochs)
    ├── best_model.keras
    ├── logs/                (TensorBoard)
    ├── curves_epoch[n].png  (plots intermédiaires)
    └── report_visuals.png   (graphiques rapport)
  Global : runs/all_runs_summary.csv



wandb: 
wandb: Run history:
wandb:            epoch ▁▁▂▂▂▂▂▂▂▃▃▃▃▃▃▄▄▄▄▄▅▅▅▅▅▆▆▆▆▆▆▆▆▇▇▇▇███
wandb:               lr ▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
wandb:       train_loss █▇▅▃▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
wandb: train_policy_acc ▁▁▂▂▃▇▇▇▇██▇████████████████████████████
wandb:  train_value_mae █▃▅▃▃▃▃▃▃▄▃▃▃▄▂▄▃▂▄▃▃▃▃▃▃▃▆▄▂▃▄▃▄▄▅▃▄▃▁▄
wandb:         val_loss ▇█▃▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
wandb:   val_policy_acc ▁▁▇▇▇▇▇▇▇██▇████████████████████████████
wandb: 
wandb: Run summary:
wandb:            epoch 176
wandb:               lr 0.001
wandb:       train_loss 3.47517
wandb: train_policy_acc 0.35544
wandb:  train_value_mae 0.28519
wandb:         val_loss 3.5199
wandb:   val_policy_acc 0.3515
wandb: 
wandb: 🚀 View run model4_resnet_tiny_lr0.001_seed42_20260330_012956 at: https://wandb.ai/rouibiamine7-universit-paris-dauphine-psl/go-19x19/runs/3iq3g5xd
wandb: ⭐️ View project at: https://wandb.ai/rouibiamine7-universit-paris-dauphine-psl/go-19x19
wandb: Synced 5 

# M9 : ResNet + Dilated Convolutions

In [7]:
# ═══════════════════════════════════════════════════════════════════
#  CELLULE M9 — ResNet + Dilated Convolutions | Projet Go 19×19
#  Conv(24)+BN → ResBlock(d=1) → ResBlock(d=2) → ResBlock(d=3)
#             → dual head(fc=56)
#
#  Budget :
#    Stem Conv(24, 3×3) + BN         =  6 816  ← MODIFIÉ : BN ajouté
#    ResBlock×3 no_bias+BN(zero-init)= 31 680
#    Heads (412 + 1086×56)           = 61 228
#    ─────────────────────────────────────────
#    Total                           = 99 724  ✓ < 100k
#
#  ⚠️  Nécessite la cellule BASE exécutée au préalable
# ═══════════════════════════════════════════════════════════════════

config = {
    'model_id'   : 9,
    'model_name' : 'resnet_dilated',
    'planes'     : PLANES,
    'moves'      : MOVES,
    'N'          : N,
    'epochs'     : 300,
    'batch'      : 128 * NUM_GPUS,
    'lr'         : 0.001,
    'l2'         : 0.0001,
    'seed'       : 42,
    'max_params' : 100_000,
    'patience'   : 20,
    'filters'    : 24,
    'dilations'  : [1, 2, 3],
    'dense_head' : 56,
}

def resblock_dilated(x, filters, dilation_rate, reg):
    """Conv(dilation)+BN+ReLU + Conv(dilation)+BN(zero-init) + skip + ReLU."""
    shortcut = x
    x = layers.Conv2D(filters, (3, 3), padding='same', use_bias=False,
                      dilation_rate=dilation_rate,
                      kernel_regularizer=reg)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.Conv2D(filters, (3, 3), padding='same', use_bias=False,
                      dilation_rate=dilation_rate,
                      kernel_regularizer=reg)(x)
    x = layers.BatchNormalization(
            gamma_initializer='zeros'  # ← AJOUTÉ : zero-init, bloc démarre
        )(x)                           #   comme identité, stabilise FP16
    x = layers.Add()([x, shortcut])
    x = layers.Activation('relu')(x)
    return x

def build_model(config):
    reg = regularizers.l2(config['l2'])
    f   = config['filters']
    dh  = config['dense_head']

    inp = keras.Input(shape=(19, 19, config['planes']), name='board')

    # ── Stem ─────────────────────────────────────────────────────────
    x = layers.Conv2D(f, (3, 3), padding='same', use_bias=False,
                      kernel_regularizer=reg)(inp)  # ← MODIFIÉ : activation
    x = layers.BatchNormalization()(x)              #   retirée du Conv,
    x = layers.Activation('relu')(x)               #   BN+ReLU séparés

    # ── ResBlocks avec dilatations croissantes ────────────────────────
    for d in config['dilations']:
        x = resblock_dilated(x, f, d, reg)

    # ── Policy head ──────────────────────────────────────────────────
    p = layers.Conv2D(1, (1, 1), activation='relu', kernel_regularizer=reg)(x)
    p = layers.Flatten()(p)
    p = layers.Dense(dh, activation='relu', kernel_regularizer=reg)(p)
    policy_head = layers.Dense(config['moves'], activation='softmax',
                                name='policy', kernel_regularizer=reg,
                                dtype='float32')(p)

    # ── Value head ───────────────────────────────────────────────────
    v = layers.Conv2D(1, (1, 1), activation='relu', kernel_regularizer=reg)(x)
    v = layers.Flatten()(v)
    v = layers.Dense(dh, activation='relu', kernel_regularizer=reg)(v)
    value_head = layers.Dense(1, activation='sigmoid',
                               name='value', kernel_regularizer=reg,
                               dtype='float32')(v)

    model = keras.Model(inputs=inp, outputs=[policy_head, value_head])
    model.summary()

    total_params = model.count_params()
    print(f"\n>>> Total params : {total_params:,}")
    assert total_params < config['max_params'], \
        f"CONTRAINTE VIOLÉE : {total_params:,} > {config['max_params']:,} !"
    print(f">>> Contrainte < {config['max_params']:,} params : ✓\n")

    model.compile(
        optimizer=keras.optimizers.Adam(
            learning_rate=config['lr'],
            clipnorm=1.0              # ← AJOUTÉ : évite NaN explosion en FP16
        ),
        loss={'policy': 'categorical_crossentropy', 'value': 'mse'},
        loss_weights={'policy': 1.0, 'value': 1.0},
        metrics={'policy': 'categorical_accuracy', 'value': 'mae'}
    )
    return model

# ── Lancement ────────────────────────────────────────────────────────
history_m9, best_loss_m9, best_acc_m9 = train_model(build_model, config)

Model: "functional_2"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ board (InputLayer)  │ (None, 19, 19,    │          0 │ -                 │
│                     │ 31)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_14 (Conv2D)  │ (None, 19, 19,    │      6,696 │ board[0][0]       │
│                     │ 24)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 19, 19,    │         96 │ conv2d_14[0][0]   │
│ (BatchNormalizatio… │ 24)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_10       │ (None, 19, 19,    │          0 │ batch_normalizat… │
│ (Activation)        │ 24)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_15 (Conv2D)  │ (None, 19, 19,    │      5,184 │ activation_10[0]… │
│                     │ 24)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 19, 19,    │         96 │ conv2d_15[0][0]   │
│ (BatchNormalizatio… │ 24)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_11       │ (None, 19, 19,    │          0 │ batch_normalizat… │
│ (Activation)        │ 24)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_16 (Conv2D)  │ (None, 19, 19,    │      5,184 │ activation_11[0]… │
│                     │ 24)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 19, 19,    │         96 │ conv2d_16[0][0]   │
│ (BatchNormalizatio… │ 24)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_3 (Add)         │ (None, 19, 19,    │          0 │ batch_normalizat… │
│                     │ 24)               │            │ activation_10[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_12       │ (None, 19, 19,    │          0 │ add_3[0][0]       │
│ (Activation)        │ 24)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_17 (Conv2D)  │ (None, 19, 19,    │      5,184 │ activation_12[0]… │
│                     │ 24)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 19, 19,    │         96 │ conv2d_17[0][0]   │
│ (BatchNormalizatio… │ 24)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_13       │ (None, 19, 19,    │          0 │ batch_normalizat… │
│ (Activation)        │ 24)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_18 (Conv2D)  │ (None, 19, 19,    │      5,184 │ activation_13[0]… │
│                     │ 24)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 19, 19,    │         96 │ conv2d_18[0][0]   │
│ (BatchNormalizatio… │ 24)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_4 (Add)         │ (None, 19, 19,    │          0 │ batch_normalizat

 Total params: 99,700 (389.45 KB)

 Trainable params: 99,364 (388.14 KB)

 Non-trainable params: 336 (1.31 KB)

...

2026-03-30 06:33:24.599194: E tensorflow/core/framework/node_def_util.cc:680] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_16}}
2026-03-30 06:33:31.476605: E tensorflow/core/framework/node_def_util.cc:680] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),

  [EarlyStopping] patience 20/20
  [EarlyStopping] Arrêt à l'époque 242.


wandb: updating run metadata


✅ report_visuals.png sauvegardé
  [DRIVE] Sync final → /kaggle/working/go_project3/model9_resnet_dilated_lr0.001_seed42_20260330_033551

  RUN TERMINÉ : model9_resnet_dilated_lr0.001_seed42_20260330_033551
  Drive       : /kaggle/working/go_project3/model9_resnet_dilated_lr0.001_seed42_20260330_033551
  Best loss        : 3.4456
  Best policy acc  : 0.3583
  Epochs réelles   : 242/300
  Params           : 99,700
  Outputs          : runs/model9_resnet_dilated_lr0.001_seed42_20260330_033551/
    ├── config.txt
    ├── history.csv          (train, toutes les epochs)
    ├── val_results.csv      (val fixe, tous les 10 epochs)
    ├── best_model.keras
    ├── logs/                (TensorBoard)
    ├── curves_epoch[n].png  (plots intermédiaires)
    └── report_visuals.png   (graphiques rapport)
  Global : runs/all_runs_summary.csv



wandb: uploading history steps 241-241, summary, console lines 3482-3504
wandb: 
wandb: Run history:
wandb:            epoch ▁▁▁▁▂▂▂▂▂▃▃▃▃▃▃▄▄▄▄▄▄▄▅▅▅▅▅▅▆▆▆▇▇▇▇▇▇███
wandb:               lr ▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
wandb:       train_loss █▆▄▃▃▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
wandb: train_policy_acc ▁▃▇▇▇▇▇█████████████████████████████████
wandb:  train_value_mae █▄▃▃▄▃▃▄▃▂▄▄▃▃▃▃▄▄▁▃▂▃▄▅▃▄▄▅▃▂▃▄▄▄▃▃▄▂▂▄
wandb:         val_loss ██▃▂▂▂▂▁▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
wandb:   val_policy_acc ▁▆▇▇▇▇██████████████████████████████████
wandb: 
wandb: Run summary:
wandb:            epoch 242
wandb:               lr 0.001
wandb:       train_loss 3.43738
wandb: train_policy_acc 0.35648
wandb:  train_value_mae 0.28658
wandb:         val_loss 3.46407
wandb:   val_policy_acc 0.34946
wandb: 
wandb: 🚀 View run model9_resnet_dilated_lr0.001_seed42_20260330_033551 at: https://wandb.ai/rouibiamine7-universit-paris-dauphine-psl/go-19x19/runs/pag0fjpw
wandb: ⭐️ View project at: https:

# M10 : CNN Transformer

In [8]:
# ═══════════════════════════════════════════════════════════════════
#  CELLULE M10 — CNN + Transformer Encoder | Projet Go 19×19
#  Conv(20)×2 → PosEmb(361,20) → [LN+MHA+skip] → [LN+FFN+skip]
#             → Reshape(19,19,20) → dual head(fc=70)
#
#  Budget :
#    CNN Conv(20)×2                  =  9 220  ← MODIFIÉ : 16→20 filtres
#    Pos embedding (361×20)          =  7 220  ← AJOUTÉ
#    LN pre-attn (20)                =     40
#    MHA(heads=4, key_dim=16)        =  5 332
#    LN pre-FFN (20)                 =     40  ← AJOUTÉ
#    FFN Dense(40)+Dense(20)         =  1 660  ← AJOUTÉ
#    Heads policy+value (dh=70)      = 76 424
#    ─────────────────────────────────────────
#    Total                           = 99 936  ✓ < 100k
#
#  Fixes vs M10 original :
#    1. filters 16→20 : features plus riches pour l'attention
#    2. Positional embedding appris : le modèle sait où est chaque case
#    3. FFN après MHA : bloc Transformer complet (standard ViT/BERT)
#    4. clipnorm=1.0 : évite NaN explosion en FP16
#
#  ⚠️  Nécessite la cellule BASE exécutée au préalable
# ═══════════════════════════════════════════════════════════════════

config = {
    'model_id'   : 10,
    'model_name' : 'cnn_transformer',
    'planes'     : PLANES,
    'moves'      : MOVES,
    'N'          : N,
    'epochs'     : 300,
    'batch'      : 128 * NUM_GPUS,
    'lr'         : 0.001,
    'l2'         : 0.0001,
    'seed'       : 42,
    'max_params' : 100_000,
    'patience'   : 20,
    'filters'    : 20,
    'mha_heads'  : 4,
    'key_dim'    : 16,
    'dense_head' : 70,
}

def build_model(config):
    reg  = regularizers.l2(config['l2'])
    f    = config['filters']
    dh   = config['dense_head']
    seq  = 19 * 19              # 361 tokens

    inp = keras.Input(shape=(19, 19, config['planes']), name='board')

    # ── CNN Backbone — extraction de features locales ─────────────────
    x = layers.Conv2D(f, (3, 3), padding='same', activation='relu',
                      kernel_regularizer=reg)(inp)
    x = layers.Conv2D(f, (3, 3), padding='same', activation='relu',
                      kernel_regularizer=reg)(x)

    # ── Reshape : (19,19,20) → (361,20) pour le Transformer ──────────
    x = layers.Reshape((seq, f))(x)

    # ── Positional Embedding appris ───────────────────────────────────
    # ← AJOUTÉ : sans ça l'attention ignore la position spatiale des cases
    positions = tf.range(seq)
    pos_emb   = layers.Embedding(seq, f)(positions)  # (361, 20)
    x         = x + pos_emb

    # ── Transformer Encoder Block complet (pre-norm style) ────────────
    # Sous-bloc 1 : Self-Attention
    residual = x
    x = layers.LayerNormalization()(x)
    x = layers.MultiHeadAttention(
            num_heads=config['mha_heads'],
            key_dim=config['key_dim']
        )(x, x)
    x = layers.Add()([x, residual])

    # Sous-bloc 2 : FFN ← AJOUTÉ : transformation non-linéaire
    residual2 = x
    x = layers.LayerNormalization()(x)
    x = layers.Dense(f * 2, activation='relu')(x)   # expand : 20→40
    x = layers.Dense(f)(x)                          # project : 40→20
    x = layers.Add()([x, residual2])

    # ── Reshape : (361,20) → (19,19,20) pour les têtes Conv ──────────
    x = layers.Reshape((19, 19, f))(x)

    # ── Policy head ──────────────────────────────────────────────────
    p = layers.Conv2D(1, (1, 1), activation='relu', kernel_regularizer=reg)(x)
    p = layers.Flatten()(p)
    p = layers.Dense(dh, activation='relu', kernel_regularizer=reg)(p)
    policy_head = layers.Dense(config['moves'], activation='softmax',
                                name='policy', kernel_regularizer=reg,
                                dtype='float32')(p)

    # ── Value head ───────────────────────────────────────────────────
    v = layers.Conv2D(1, (1, 1), activation='relu', kernel_regularizer=reg)(x)
    v = layers.Flatten()(v)
    v = layers.Dense(dh, activation='relu', kernel_regularizer=reg)(v)
    value_head = layers.Dense(1, activation='sigmoid',
                               name='value', kernel_regularizer=reg,
                               dtype='float32')(v)

    model = keras.Model(inputs=inp, outputs=[policy_head, value_head])
    model.summary()

    total_params = model.count_params()
    print(f"\n>>> Total params : {total_params:,}")
    assert total_params < config['max_params'], \
        f"CONTRAINTE VIOLÉE : {total_params:,} > {config['max_params']:,} !"
    print(f">>> Contrainte < {config['max_params']:,} params : ✓\n")

    model.compile(
        optimizer=keras.optimizers.Adam(
            learning_rate=config['lr'],
            clipnorm=1.0              # ← AJOUTÉ : évite NaN explosion en FP16
        ),
        loss={'policy': 'categorical_crossentropy', 'value': 'mse'},
        loss_weights={'policy': 1.0, 'value': 1.0},
        metrics={'policy': 'categorical_accuracy', 'value': 'mae'}
    )
    return model

# ── Lancement ────────────────────────────────────────────────────────
history_m10, best_loss_m10, best_acc_m10 = train_model(build_model, config)

Model: "functional_3"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ board (InputLayer)  │ (None, 19, 19,    │          0 │ -                 │
│                     │ 31)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_23 (Conv2D)  │ (None, 19, 19,    │      5,600 │ board[0][0]       │
│                     │ 20)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_24 (Conv2D)  │ (None, 19, 19,    │      3,620 │ conv2d_23[0][0]   │
│                     │ 20)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ reshape (Reshape)   │ (None, 361, 20)   │          0 │ conv2d_24[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_6 (Add)         │ (None, 361, 20)   │          0 │ reshape[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalization │ (None, 361, 20)   │         40 │ add_6[0][0]       │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multi_head_attenti… │ (None, 361, 20)   │      5,332 │ layer_normalizat… │
│ (MultiHeadAttentio… │                   │            │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_7 (Add)         │ (None, 361, 20)   │          0 │ multi_head_atten… │
│                     │                   │            │ add_6[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, 361, 20)   │         40 │ add_7[0][0]       │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_6 (Dense)     │ (None, 361, 40)   │        840 │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_7 (Dense)     │ (None, 361, 20)   │        820 │ dense_6[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_8 (Add)         │ (None, 361, 20)   │          0 │ dense_7[0][0],    │
│                     │                   │            │ add_7[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ reshape_1 (Reshape) │ (None, 19, 19,    │          0 │ add_8[0][0]       │
│                     │ 20)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_25 (Conv2D)  │ (None, 19, 19, 1) │         21 │ reshape_1[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_26 (Conv2D)  │ (None, 19, 19, 1) │         21 │ reshape_1[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flatten_6 (Flatten) │ (None, 361)       │          0 │ conv2d_25[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flatten_7 (Flatten) │ (None, 361)       │          0 │ conv2d_26[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_8 (Dense)     │ (None, 70)        │     25,340 │ flatten_6[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_9 (Dense)     │ (None, 70)        │     25,340 │ flatten_7[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ policy (Dense)      │ (None, 361)       │     25,631 │ dense_8[0][0]     │
├─────────────────────┼───────────────────┼────────────┼─────────────────

 Total params: 92,716 (362.17 KB)

 Trainable params: 92,716 (362.17 KB)

 Non-trainable params: 0 (0.00 B)

...

2026-03-30 07:35:52.893333: E tensorflow/core/framework/node_def_util.cc:680] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_16}}
2026-03-30 07:36:02.498178: E tensorflow/core/framework/node_def_util.cc:680] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),

  [EarlyStopping] patience 20/20
  [EarlyStopping] Arrêt à l'époque 72.


wandb: updating run metadata


✅ report_visuals.png sauvegardé
  [DRIVE] Sync final → /kaggle/working/go_project3/model10_cnn_transformer_lr0.001_seed42_20260330_063347

  RUN TERMINÉ : model10_cnn_transformer_lr0.001_seed42_20260330_063347
  Drive       : /kaggle/working/go_project3/model10_cnn_transformer_lr0.001_seed42_20260330_063347
  Best loss        : 6.0001
  Best policy acc  : 0.0036
  Epochs réelles   : 72/300
  Params           : 92,716
  Outputs          : runs/model10_cnn_transformer_lr0.001_seed42_20260330_063347/
    ├── config.txt
    ├── history.csv          (train, toutes les epochs)
    ├── val_results.csv      (val fixe, tous les 10 epochs)
    ├── best_model.keras
    ├── logs/                (TensorBoard)
    ├── curves_epoch[n].png  (plots intermédiaires)
    └── report_visuals.png   (graphiques rapport)
  Global : runs/all_runs_summary.csv



wandb: uploading history steps 71-71, summary, console lines 1033-1056
wandb: 
wandb: Run history:
wandb:            epoch ▁▁▁▂▂▂▂▃▃▃▃▃▃▃▄▄▄▄▅▅▅▅▅▅▅▅▆▆▆▆▇▇▇▇▇▇▇▇██
wandb:               lr ▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
wandb:       train_loss █▃▃▃▂▂▂▂▂▂▂▂▁▁▁▁▁▁▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
wandb: train_policy_acc ▃▃▇▆▃▅▁▂▄█▂▇▇█▆▄▅▅▇▇▄▇▅▄▃▁▄▇█▅▄▅▄▂▇▆▁▃▅▃
wandb:  train_value_mae ▄▂▆▅▆▅▄▃▄▆▂█▂▅▃▅▄▆▆▆▆▆▆▄▄▁▆▅▁▆▅▅▆▃█▅▇▆▃▅
wandb:         val_loss █▅▄▃▃▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
wandb:   val_policy_acc ▄▁▁▄▃▂█▂▅▄▃▂▂▄▅▁▁▅▅▂▄▂▂▅▃▅▅▅▅▅▆▆▄▅▇▄▅▅▆▅
wandb: 
wandb: Run summary:
wandb:            epoch 72
wandb:               lr 0.001
wandb:       train_loss 5.9994
wandb: train_policy_acc 0.00298
wandb:  train_value_mae 0.29264
wandb:         val_loss 6.00036
wandb:   val_policy_acc 0.00322
wandb: 
wandb: 🚀 View run model10_cnn_transformer_lr0.001_seed42_20260330_063347 at: https://wandb.ai/rouibiamine7-universit-paris-dauphine-psl/go-19x19/runs/77dop7gz
wandb: ⭐️ View project at: https://